source: https://github.com/abhinav-kimothi/A-Simple-Guide-to-RAG/blob/main/Chapters/Chapter-03/indexing_pipeline.ipynb

# 1. Indexing Pipeline

This chapter elaborates on the four components of the indexing pipeline:
- Data loading – Responsible for connecting to external sources, and extracting and parsing information 
- Data splitting – Breaking down the text into chunks, enhancing the system’s ability to process and analyze information efficiently.
- Data conversion(embeddings) – The textual data must be converted to a numerical format, embeddings speciffically. 
- Data storage – Once the data is transform on embeddings , that must be stored in persistent memory so that the real time generation pipeline can access data whenever a - user ask a question. Vector databases are best suited for search and retrieval of embeddings.


## 1.1. Data Loading

The first step toward building a knowledge base (or non-parametric memory) of a RAG system is to source data from its original location. This data may be in the form of Word documents, PDF files, CSV, HTML, and similar. Furthermore, the data may be stored in file, block, or object stores, in data lakes, data warehouses, or even in third-party sources that can be accessed via the open internet. This process of sourcing data from its original location is called data loading. Loading documents from a list of sources may turn out to be a complicated proces

LangChain is a framework for building agents and LLM-powered applications. It helps you chain together interoperable components and third-party integrations to simplify AI application development — all while future-proofing decisions as the underlying technology evolves.

All requirement are include on requierement.txt

In [ ]:
%pip install -r requirements.txt
#%pip install langchain-community


In [ ]:
from dotenv import load_dotenv
import os

if load_dotenv():
    print("Success: .env file found with some environment variables")
else:
    print("Caution: No environment variables found. Please create .env file in the root directory or add environment variables in the .env file")


We will take the phi3 open source llm model. If we ask for the 2024 UEFA European Championship winner:

In [ ]:
from langchain_community.chat_models import ChatOllama

llm = ChatOllama(model="phi3", base_url="http://localhost:11434")

llm.invoke("Who won the 2024 UEFA European Championship?")

The phi3 knowledge cutoff was in April 2nd, 2023

We will use the AsyncHtmlLoader function from the document_loaders library in the langchain-community package. To run AsyncHtmlLoader, we will have to install another Python package called bs4:

In [2]:
#Installing bs4 package
#%pip install bs4==0.0.2 --quiet

#Importing the AsyncHtmlLoader
from langchain_community.document_loaders import AsyncHtmlLoader

#This is the URL of the Wikipedia page on the UEFA European Championship
url="https://en.wikipedia.org/wiki/UEFA_Euro_2024"

#Invoking the AsyncHtmlLoader
loader = AsyncHtmlLoader (url)

#Loading the extracted information
data = loader.load()

/home/flopez/TFM/RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.
Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.47it/s]


The metadata provide context information.This can include a data summary; the way the data was created; who created it and why; when it was created; and the size, quality, and condition of the data. 

Now clean the html data using the Html2Text­Transformer function from the document_transformers library in the langchain-community package. For Html2TextTransformer, we will also have to install another package called html2text.

In [ ]:
#Install html2text
#%pip install html2text==2024.2.26 –quiet

#Import Html2TextTransformer
from langchain_community.document_transformers import Html2TextTransformer

#Assign the Html2TextTransformer function
html2text = Html2TextTransformer()

#Call transform_documents
html_data_transformed = html2text.transform_documents(data)

#print(html_data_transformed[0].page_content)

The output of the page_content is now free of any HTML tags and contains only the text from the webpage.

The source for our data was Wikipedia (more precisely, a web address pointing to a Wikipedia page), and the format was HTML. The source can also be other storage locations such as AWS S3, SQL/NoSQL databases, Google Drive, GitHub, even WhatsApp, YouTube, and other social media sites. Likewise, the data formats can be .doc, .pdf, .csv, .ppt, .eml, and the like. Most of the time, you will be able to use frameworks such as LangChain that have integrations for the sources and the formats already built in. Sometimes, you may have to build custom connectors and loaders.

## 1.2. Data splitting

Breaking down long pieces of text to manageable segments is called data splitting or chunking. 

1. Advantages of chunking: Due to the inherent nature of the technology, the number of tokens (loosely, words) LLMs can work with at a time is limited. The way to address this problem is to break the text down into smaller chunks.
2. Lost-in-the-middle-problem: Lost-in-the-middle problem—Even in those LLMs that have a long context window (e.g., Claude 3 by Anthropic has a context window of up to 200,000 tokens), a problem with accurately reading the information has been observed. It has been noticed that accuracy declines dramatically if the relevant information is somewhere in the middle of the prompt. This problem can be addressed by passing only the relevant information to the LLM instead of the entire document.
3. Ease of search: This is not a problem with the LLM per se, but it has been observed that large chunks of text are harder to search over. When we use a retriever (recall the generation pipeline introduced in chapter 2), it is more efficient to search over smaller pieces of text.


Chunking aims to keep meaningful data together. If we are dealing with data in the form of HTML, Markdown, JSON, or even computer code, it makes more sense to split the data based on the structure rather than a fixed size.

HTML is organized in headers and sections. For such formats, a specialized chunking approach can be employed. LangChain offers classes such as MarkdownHeaderTextSplitter, HTMLHeader­TextSplitter, and RecursiveJsonSplitter, among others, for these formats. 

In [3]:
#Installing lxml
#%pip install lxml==5.3.1 --quiet

# Import the HTMLHeaderTextSplitter library
from langchain_text_splitters import HTMLSectionSplitter

# Set URL as the Wikipedia page link
url="https://en.wikipedia.org/wiki/UEFA_Euro_2024"

loader = AsyncHtmlLoader (url)

html_data = loader.load()

# Specify the header tags on which splits should be made
sections_to_split_on=[
    ("h1", "Header 1"),
    ("h2", "Header 2"),
    ("table ", "Table"),
    ("p", "Paragraph")
]

# Create the HTMLHeaderTextSplitter function
splitter = HTMLSectionSplitter(sections_to_split_on)

# Create splits in text obtained from the URL
split_content = splitter.split_text(html_data[0].page_content)

Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.79it/s]


In [4]:
split_content

[Document(metadata={'Header 1': '#TITLE#'}, page_content='Jump to content \n \n \n \n \n \n \n \n Main menu \n \n \n \n \n \n Main menu \n move to sidebar \n hide \n \n \n \n\t\tNavigation\n\t \n \n \n Main page \n Contents \n Current events \n Random article \n About Wikipedia \n Contact us \n \n \n \n \n \n\t\tContribute\n\t \n \n \n Help \n Learn to edit \n Community portal \n Recent changes \n Upload file \n Special pages \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n Search \n \n \n \n \n \n \n \n \n \n \n \n Search \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n Appearance \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n Donate \n \n \n Create account \n \n \n Log in \n \n \n \n \n \n \n \n \n Personal tools \n \n \n \n \n \n \n \n Donate \n \n \n \n Create account \n \n \n \n Log in \n \n \n \n \n \n \n \n \n \n \n \n \n \n  CentralNotice'),
 Document(metadata={'Header 2': 'Contents'}, page_content='Contents \n move to sidebar \n hide \n

In [ ]:
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_community.document_transformers import Html2TextTransformer

url = "https://en.wikipedia.org/wiki/UEFA_Euro_2024"

loader = AsyncHtmlLoader([url])
docs = loader.load()

transformer = Html2TextTransformer()
clean_docs = transformer.transform_documents(docs)

In [ ]:
clean_docs

In [ ]:
split_content

In [ ]:
len(split_content)

We can further chunk these 45 chunks using a fixed-size chunking method such as RecursiveCharacterTextSplitter.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,      # 🔥 smaller = better for Phi-3
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " "]
)

final_chunks = text_splitter.split_documents(clean_docs)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
separators=["\n\n","\n","."],
chunk_size=1000, chunk_overlap=100, 
)

final_chunks = text_splitter.split_documents(split_content)

In [ ]:
len(final_chunks)

## 1.3. Data conversion (embeddings)

For a computer to process any kind of nonnumeric data such as text or image, it must be first converted into a numerical form. 

Embeddings is a design pattern that is extremely helpful in the fields of data science, machine learning, and AI. Embeddings are vector representations of data. As a general definition, embeddings are data that has been transformed into n-dimensional matrixes. The word embedding is a vector representation of words.

Code example for creating embeddings using an open source embeddings model all-MPnet-base-v2 via Hugging Face: 

In [ ]:
#%pip install langchain-huggingface
# Import HuggingFaceEmbeddings from embeddings library
from langchain_huggingface import HuggingFaceEmbeddings

# Instantiate the embeddings model. The embeddings model_name can be changed as desired
embeddings = HuggingFaceEmbeddings(
model_name="sentence-transformers/all-mpnet-base-v2"
)

# Create embeddings for all chunks
hf_embeddings = embeddings.embed_documents(
[chunk.page_content for chunk in final_chunks]
)

#Check the length(dimension) of the embedding
len(hf_embeddings [0])

## 1.4.Storage (vector databases)

The data has been loaded, split, and converted into embeddings. For us to use this information repeatedly, we need to store it in memory so that it can be accessed on demand. Vector Databases are built to handle high dimensional vectors. These databases specialize in indexing and storing vector embeddings for fast semantic search and retrieval.

We will use a vector index db such as Facebook AI Similarity Search (FAISS), which is supported by LangChain. To use FAISS, we first must install the faiss-cpu library. 

In [ ]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

index = faiss.IndexFlatIP(len(hf_embeddings[0]))

# Instantiate the FAISS object
vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

# Add the chunks
vector_store.add_documents(documents=final_chunks)

# Check the number of chunks that have been indexed
vector_store.index.ntotal

We can also save the vector store in persistent memory to use it later!

In [ ]:
vector_store.save_local(folder_path="Assets/Data",index_name="CWC_index")

In [ ]:
# Original Question
query = "Who won the 2024 UEFA European Championship?"
# Ranking the chunks in descending order of similarity
docs = vector_store.similarity_search(query)
# Printing the top ranked chunk
print(docs[0].page_content)

Similarity search orders the chunks in descending order of similarity, meaning that the most similar chunks to the query are ranked on top. In the previous example, we can observe that the chunk that speaks about the world cup final has been ranked on top.

# 2. Generation Pipeline

The generation pipeline involves three processes: retrieval, augmentation, and generation. The retrieval process is responsible for fetching the information relevant to the user query from the knowledge base. Augmentation is the process of combining the fetched information with the user query. Generation is the last step, in which the LLM generates a response based on the augmented prompt. This chapter discusses these three processes in detail. 

### 2.1 Retrieval

In [ ]:
# Import FAISS class from vectorstore library
from langchain_community.vectorstores import FAISS

# Load the database stored in the local directory
vector_store=FAISS.load_local(
folder_path="Assets/Data", 
index_name="CWC_index",
embeddings=embeddings, 
allow_dangerous_deserialization=True
)

# Original Question
query = "Who won the 2024 UEFA European Championship?"

# Ranking the chunks in descending order of similarity
retrieved_docs = vector_store.similarity_search(query, k=2)

In [ ]:
import textwrap

for i, doc in enumerate(retrieved_docs):
    print(textwrap.fill(f"\nRetrieved Chunk {i+1}:\n{doc.page_content}",width=100))
    print("\n\n")


### 2.3. Augmentation

In [ ]:

# taking first two retrieved documents
retrieved_context=retrieved_docs[0].page_content + retrieved_docs[1].page_content

# Creating the prompt
augmented_prompt=f"""

Given the context below answer the question.

Question: {query} 

Context : {retrieved_context}

Remember to answer only based on the context provided and not from any other source. 

If the question cannot be answered based on the provided context, say I don’t know.

"""

print(textwrap.fill(augmented_prompt,width=150))



## 2.4. Generation

In [ ]:
from langchain_community.chat_models import ChatOllama

# Set up LLM
llm = ChatOllama(
    model="phi3",
    temperature=0,
    base_url="http://localhost:11434"  # change if needed
)

messages = [
    ("human", augmented_prompt)
]

ai_msg = llm.invoke(messages)

# Extract answer
answer = ai_msg.content
print(answer)